In [5]:
from langchain.chat_models import init_chat_model 
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

from dotenv import load_dotenv 

load_dotenv()

model_1 = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)

model_2 = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.0,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)

Unexpected argument 'load_env' provided to ChatGoogleGenerativeAI.
/home/lakshay/LangChain/.venv/lib/python3.12/site-packages/langchain/chat_models/base.py:540: UserWarning: WARNING! load_env is not default parameter.
                load_env was transferred to model_kwargs.
                Please confirm that load_env is what you intended.
  return creator_func(model=model, **kwargs)
Unexpected argument 'load_env' provided to ChatGoogleGenerativeAI.


In [7]:
prompt1 = PromptTemplate(
    template = "Generate short and simple notes on the topic {topic}",
    input_variables = ["topic"]
)

prompt2 = PromptTemplate(
    template = "Generate 5 quiz questions based on the following topic {topic}",
    input_variables = ["notes"]
)

prompt3 = PromptTemplate(
    template = "Merge the following notes and quiz questions into a single document: {notes} {quiz_questions}",
    input_variables = ["notes", "quiz_questions"]
)

In [8]:
parser = StrOutputParser()

In [9]:
parallel_chain = RunnableParallel({

    'notes' : prompt1 | model_1 | parser,
    'quiz_questions' : prompt2 | model_2 | parser

})

merge_chain = prompt3 | model_1 | parser

chain = parallel_chain | merge_chain   


In [10]:
chain.invoke({"topic": "Artificial Intelligence"})

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


'Here is a single document merging the notes and quiz questions on Artificial Intelligence:\n\n---\n\n# Understanding Artificial Intelligence: Notes and Quiz\n\n## Short and Simple Notes on Artificial Intelligence\n\nArtificial Intelligence ('

In [11]:
chain.get_graph().print_ascii()

                +-------------------------------------+                
                | Parallel<notes,quiz_questions>Input |                
                +-------------------------------------+                
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati